# 🧪 Linting System Tests

### 🔁 Step 1: Set up Environment

In [1]:
from pathlib import Path
import os, sys

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_scores import seed_score_providers
from app.db.seeders.seed_tools import seed_tool_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.seeders.seed_agent_engine_providers import seed_agent_engine_providers
from app.db.seeders.seed_agent_providers import seed_agent_providers
from app.db.seeders.seed_state_providers import seed_state_providers
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)
    seed_context_providers(session)
    seed_agent_engine_providers(session)
    seed_agent_providers(session)
    seed_state_providers(session)

Working directory is now: C:\Repos\codecritic
Seeded AgentPrompt 'generate' with GUID: 1218bdad-a9eb-431b-9c14-820f6e3f4fdd
Seeded AgentPrompt 'linting_generator_agent' with GUID: fcdc73dd-ef72-4d0c-92a4-985a44fe17f7
Seeded SystemPrompt 'format' with GUID: ece05ec9-9848-4634-9834-ed3c908b515e
Seeded SystemPrompt 'linting_system' with GUID: 95bade06-1f10-4c4c-895b-af08fef963c8
Seeded prompt provider configurations successfully.
Seeded tool configurations successfully.
Seeded score providers successfully.
Seeded context provider configurations successfully.
Seeded agent engine configurations successfully.
Seeded agent provider configurations successfully.
Seeded state provider configurations successfully.


### 🔍 Step 2: Create Basic Prompt

In [2]:
from sqlalchemy.orm import Session
from app.db.models import AgentPrompt, SystemPrompt, PromptProviderConfig
from app.factories.prompt_provider_factory import PromptProviderFactory

# Open DB session
with Session(bind=engine) as session:
    # 🔍 Fetch records by name
    system_prompt = session.query(SystemPrompt).filter_by(name="linting_system").first()
    agent_prompt = session.query(AgentPrompt).filter_by(name="linting_generator_agent").first()
    provider_record = session.query(PromptProviderConfig).filter_by(name="linting_prompt_provider").first()

    # 🧱 Validate records exist
    assert system_prompt, "❌ Missing system prompt: linting_system"
    assert agent_prompt, "❌ Missing agent prompt: linting_generator_agent"
    assert provider_record, "❌ Missing prompt provider: linting_prompt_provider"

    # ⚙️ Load provider from module path
    prompt_provider = PromptProviderFactory.create(provider_record.id)

    # 🚀 Run provider with prompt file paths
    merged_prompt = prompt_provider.run(
        session_id="notebook-dev-session",
        input={
            "system_prompt_path": system_prompt.artifact_path,
            "agent_prompt_path": agent_prompt.artifact_path
    }
)

    # 📋 Output final combined prompt
    print("🧠 Combined Prompt:\n\n" + merged_prompt)


🧠 Combined Prompt:

            You are the Generator Agent in the Linting System.

You specialize in incrementally improving Python code quality by addressing lint violations, enhancing readability, and promoting best practices.

Your strategy is:
- Focus on minimal, targeted improvements that preserve original structure
- Apply type hints and clarify variable names where appropriate
- Use reasoning to resolve tool-reported issues when they conflict

You will be evaluated by a Discriminator Agent and may be prompted to revise your output. If you are in a retry round, incorporate prior feedback before retrying.

Maintain a helpful, confident tone in your comments. Your improvements should be easy for a human developer to understand and accept.

            ---

            You are operating within the Linting System, which enforces strict Python code quality and formatting compliance.

All Generator Agents must:
- Conform to Python PEP8 guidelines
- Use tools such as `black`, `ruff`, `

### 🔍 Step 3: Create Score Provider

In [3]:
from sqlalchemy.orm import Session
from app.db.models import ToolProviderConfig, ScoreProviderConfig
from app.factories.score_provider_factory import ScoreProviderFactory

required_tools = ["ruff", "mypy", "black", "radon"]

# Step 1: Seed tool IDs into score provider config
with Session(bind=engine) as session:
    tool_rows = session.query(ToolProviderConfig).filter(
        ToolProviderConfig.name.in_(required_tools)
    ).all()

    tool_ids = {tool.name: tool.id for tool in tool_rows}  # ✅ use .id, not .guid
    assert all(t in tool_ids for t in required_tools), f"Missing tools: {set(required_tools) - set(tool_ids)}"

    score_row = session.query(ScoreProviderConfig).filter_by(name="linting_score_provider").first()
    assert score_row, "❌ Score provider not found"

    score_row.config = {"tool_ids": tool_ids}
    session.add(score_row)
    session.commit()
    score_id = score_row.id  # ✅ store ID here before session closes

# Step 2: Use a fresh session context for factory
score_provider = ScoreProviderFactory.create(score_id)

# Step 3: Run the tool
result = score_provider.run(
    input={"file_path": r"C:\Repos\codecritic\tests\notebooks\system_development\bad_code.py"},
    session_id="notebook-dev-session"
)

print("🎯 Final Score:", result.value)
print("🔬 Components:")
for k, v in result.components.items():
    print(f"  • {k}: {v}")


🎯 Final Score: 0.855
🔬 Components:
  • ruff: 0.75
  • mypy: 0.95
  • black: 1.0
  • radon: 0.7


### 🔍 Step 4: Create Context Provider

In [4]:
from sqlalchemy.orm import Session
from app.db.models import ContextProviderConfig
from app.factories.context_provider_factory import ContextProviderFactory

# Step 1: Get the context provider config (already seeded)
with Session(bind=engine) as session:
    context_row = session.query(ContextProviderConfig).filter_by(name="linting_context_provider").first()
    assert context_row, "❌ Context provider not found: linting_context_provider"
    context_id = context_row.id

# Step 2: Create the context provider (inject score provider and engine)
context_provider = ContextProviderFactory.create(
    context_id,
    score_provider=score_provider,
    engine=engine  # ✅ critical for DB-bound log access
)

# Step 3: Run the context provider
context_output = context_provider.run(
    input={
        "file_path": r"C:\Repos\codecritic\tests\notebooks\system_development\bad_code.py",
        "session_id": "notebook-dev-session",
        "system": "linting"
    },
    session_id="notebook-dev-session"
)

print("🧠 Context Generated:")
print(context_output)


🧠 Context Generated:
{
  "file_path": "C:\\Repos\\codecritic\\tests\\notebooks\\system_development\\bad_code.py",
  "source_code": "import sys, os\n\n\ndef add(a, b):\n    result = a + b\n    return result\n\n\ndef unused_function():\n    print(\"I am never called\")\n\n\nclass myClass:\n    def __init__(self, value):\n        self.Value = value\n\n    def printvalue(self):\n        print(self.Value)\n\n\nadd(1, 2)\n",
  "score": {
    "name": "linting_score",
    "value": 0.855,
    "components": {
      "ruff": 0.75,
      "mypy": 0.95,
      "black": 1.0,
      "radon": 0.7
    }
  },
  "conversation_log": []
}


In [5]:
from sqlalchemy.orm import Session
from app.db.models import AgentPrompt, SystemPrompt, PromptProviderConfig
from app.factories.prompt_provider_factory import PromptProviderFactory
import json

with Session(bind=engine) as session:
    system_prompt = session.query(SystemPrompt).filter_by(name="linting_system").first()
    agent_prompt = session.query(AgentPrompt).filter_by(name="linting_generator_agent").first()
    prompt_config = session.query(PromptProviderConfig).filter_by(name="linting_prompt_provider").first()
    assert system_prompt and agent_prompt and prompt_config, "Missing one or more prompt components"
    prompt_provider = PromptProviderFactory.create(prompt_config.id)

# Pass in parsed context + prompt paths
parsed_context = json.loads(context_output)
final_prompt = prompt_provider.run(
    session_id="notebook-dev-session",
    input={
        "system_prompt_path": system_prompt.artifact_path,
        "agent_prompt_path": agent_prompt.artifact_path,
        "context": parsed_context
    }
)

print("🧠 Final Prompt:")
print(final_prompt)


🧠 Final Prompt:
            You are the Generator Agent in the Linting System.

You specialize in incrementally improving Python code quality by addressing lint violations, enhancing readability, and promoting best practices.

Your strategy is:
- Focus on minimal, targeted improvements that preserve original structure
- Apply type hints and clarify variable names where appropriate
- Use reasoning to resolve tool-reported issues when they conflict

You will be evaluated by a Discriminator Agent and may be prompted to revise your output. If you are in a retry round, incorporate prior feedback before retrying.

Maintain a helpful, confident tone in your comments. Your improvements should be easy for a human developer to understand and accept.

            ---

            You are operating within the Linting System, which enforces strict Python code quality and formatting compliance.

All Generator Agents must:
- Conform to Python PEP8 guidelines
- Use tools such as `black`, `ruff`, `mypy

In [6]:
from app.db.models import AgentEngineProviderConfig
from app.factories.agent_engine_provider_factory import AgentEngineProviderFactory

from dotenv import load_dotenv
load_dotenv("env/.env", override=True)


with Session(bind=engine) as session:
    engine_row = session.query(AgentEngineProviderConfig).filter_by(name="openai_gpt_4o_agent_engine").first()
    assert engine_row, "❌ Engine provider not found"
    engine_provider = AgentEngineProviderFactory.create(engine_row.id)

response = engine_provider.run(
    input={"prompt": final_prompt},
    session_id="notebook-dev-session"
)

print("🧠 GPT-4o Response:")
print(response)

🧠 GPT-4o Response:
```python
import sys
import os


def add(a: int, b: int) -> int:
    result = a + b
    return result


class MyClass:
    def __init__(self, value: int):
        self.value = value

    def print_value(self) -> None:
        print(self.value)


add(1, 2)
```

[CONVERSATION_LOG_ENTRY]
1. **Imports**: Split the combined import statement into separate lines for `sys` and `os` to adhere to PEP8 guidelines.
2. **Type Hints**: Added type hints to the `add` function and the `MyClass` constructor to improve code clarity and maintainability.
3. **Unused Function**: Removed the `unused_function` as it was not called anywhere in the code, reducing unnecessary clutter.
4. **Class Naming**: Changed `myClass` to `MyClass` to follow the PEP8 naming convention for classes.
5. **Attribute Naming**: Changed `Value` to `value` in `MyClass` to follow the PEP8 naming convention for attributes.
6. **Method Naming**: Changed `printvalue` to `print_value` to adhere to PEP8 naming conventio

In [7]:
from sqlalchemy.orm import Session
from app.db.models import AgentProviderConfig
from app.factories.agent_provider_factory import AgentProviderFactory
import json

# Load agent
with Session(bind=engine) as session:
    row = session.query(AgentProviderConfig).filter_by(name="linting_generator_agent_provider").first()
    assert row, "❌ Agent provider not found"
    agent = AgentProviderFactory.create(row.id)

# Load context from previous cell
context = json.loads(context_output)

# Run agent
response = agent.run(
    input={
        "context": context,
        "file_path": r"C:\Repos\codecritic\tests\notebooks\system_development\bad_code.py",
        "system": "linting" 
    },
    session_id="notebook-dev-session"
)

print("✅ Agent Response:")
print(response)


✅ Agent Response:
```python
import sys
import os


def add(a: int, b: int) -> int:
    result = a + b
    return result


class MyClass:
    def __init__(self, value: int) -> None:
        self.value = value

    def print_value(self) -> None:
        print(self.value)


add(1, 2)
```

[CONVERSATION_LOG_ENTRY]
- Removed the unused `unused_function` to eliminate dead code and improve clarity.
- Split the import statement into two separate lines for `sys` and `os` to comply with PEP8 guidelines.
- Added type hints to the `add` function and the `MyClass` constructor to improve code readability and maintainability.
- Renamed `myClass` to `MyClass` to follow the PEP8 naming convention for classes.
- Changed `Value` to `value` in `MyClass` to follow the PEP8 naming convention for instance variables.
- Renamed `printvalue` to `print_value` to follow the PEP8 naming convention for method names.
- Ensured all changes preserved the original functionality and intent of the code.
[/CONVERSATION_LO

In [8]:
from sqlalchemy.orm import Session
from app.db.models import StateProviderConfig
from app.db.connection import DB_PATH
from app.factories.state_provider_factory import StateProviderFactory

engine = create_engine(f"sqlite:///{DB_PATH}")

with Session(bind=engine) as session:
    discriminator_row = session.query(StateProviderConfig).filter_by(
        name="linting_discriminator_state_provider"
    ).first()

    print(f"🔎 Provider DB entry found: {discriminator_row}")

    # Explicitly check the artifact_path and provider_id
    print("Artifact Path:", discriminator_row.artifact_path)
    print("Provider ID:", discriminator_row.id)

    # Now manually create provider and inspect it
    provider = StateProviderFactory.create(discriminator_row.id)
    print("Loaded Provider Class:", provider.__class__.__name__)

    # CRITICAL FIX: Update the artifact_path if it's pointing to the wrong provider
    if "generator" in discriminator_row.artifact_path:
        discriminator_row.artifact_path = discriminator_row.artifact_path.replace("generator", "discriminator")
        session.commit()
        print("🚨 Fixed discriminator artifact_path!")


🔎 Provider DB entry found: <app.db.models.StateProviderConfig object at 0x000001196EEF9220>
Artifact Path: fdd93df6-132e-4a4c-9c83-04413e73e6b7
Provider ID: 3
Loaded Provider Class: LintingDiscriminatorStateProvider


In [10]:
from sqlalchemy.orm import Session
from app.db.models import StateProviderConfig, ContextProviderConfig
from app.factories.state_provider_factory import StateProviderFactory
from app.factories.context_provider_factory import ContextProviderFactory
import json

# Step 1: Load explicitly the generator state and context providers
with Session(bind=engine) as session:
    state_row = session.query(StateProviderConfig).filter_by(
        name="linting_generator_state_provider"
    ).first()
    context_row = session.query(ContextProviderConfig).filter_by(
        name="linting_context_provider"
    ).first()
    state_provider = StateProviderFactory.create(state_row.id)
    context_provider = ContextProviderFactory.create(
        context_row.id, score_provider=score_provider, engine=engine
    )

# Step 2: Prepare initial state and run generator transition
initial_state = {
    "state": "start",
    "file_path": r"C:\Repos\codecritic\tests\notebooks\system_development\bad_code.py",
    "context": json.loads(context_output),
    "session_id": "notebook-dev-session"
}

print("✅ Running Generator State")
state_provider.run(
    input={
        "file_path": initial_state["file_path"],
        "context": initial_state["context"],
        "system": "linting"
    },
    session_id="notebook-dev-session"
)

new_state = state_provider.transition(initial_state, session_id="notebook-dev-session")

# Step 3: Refresh context explicitly from DB
refreshed_context = context_provider.run(
    input={
        "file_path": new_state["file_path"],
        "session_id": "notebook-dev-session",
        "system": "linting"
    },
    session_id="notebook-dev-session"
)

new_state["context"] = json.loads(refreshed_context)

# Step 4: Display result
print("🧠 Generator Final State with Conversation Log:")
print(json.dumps(new_state, indent=2))


✅ Running Generator State
🧠 Generator Final State with Conversation Log:
{
  "state": "end",
  "file_path": "C:\\Repos\\codecritic\\tests\\notebooks\\system_development\\bad_code.py",
  "context": {
    "file_path": "C:\\Repos\\codecritic\\tests\\notebooks\\system_development\\bad_code.py",
    "source_code": "import sys, os\n\n\ndef add(a, b):\n    result = a + b\n    return result\n\n\ndef unused_function():\n    print(\"I am never called\")\n\n\nclass myClass:\n    def __init__(self, value):\n        self.Value = value\n\n    def printvalue(self):\n        print(self.Value)\n\n\nadd(1, 2)\n",
    "score": {
      "name": "linting_score",
      "value": 0.855,
      "components": {
        "ruff": 0.75,
        "mypy": 0.95,
        "black": 1.0,
        "radon": 0.7
      }
    },
    "conversation_log": [
      "- Removed the unused `unused_function` to eliminate dead code and improve clarity.\n- Split the import statement into two separate lines for `sys` and `os` to comply with P

In [11]:
from sqlalchemy.orm import Session
from app.db.models import StateProviderConfig, ContextProviderConfig
from app.factories.state_provider_factory import StateProviderFactory
from app.factories.context_provider_factory import ContextProviderFactory
import json

# Step 1: Explicitly load discriminator state provider and context provider
with Session(bind=engine) as session:
    state_row = session.query(StateProviderConfig).filter_by(
        name="linting_discriminator_state_provider"
    ).first()
    context_row = session.query(ContextProviderConfig).filter_by(
        name="linting_context_provider"
    ).first()

    assert state_row and context_row, "❌ Missing provider(s)"

    discriminator_state_provider = StateProviderFactory.create(state_row.id)
    context_provider = ContextProviderFactory.create(
        context_row.id, score_provider=score_provider, engine=engine
    )

# Step 2: Use the previously generated state
initial_state = new_state

# Step 3: Explicitly run discriminator provider to properly initialize internal state
print("✅ Running Discriminator State")
discriminator_state_provider.run(
    input={
        "file_path": initial_state["file_path"],
        "system": "linting"
    },
    session_id="notebook-dev-session"
)

# Step 4: Run discriminator transition
discriminator_state = discriminator_state_provider.transition(
    initial_state, session_id="notebook-dev-session"
)

# Step 5: Refresh context explicitly from DB
refreshed_context = context_provider.run(
    input={
        "file_path": discriminator_state["file_path"],
        "session_id": "notebook-dev-session",
        "system": "linting"
    },
    session_id="notebook-dev-session"
)
discriminator_state["context"] = json.loads(refreshed_context)

# Step 6: Output final discriminator FSM state and logs
print("🧠 Discriminator State Result:")
print(json.dumps(discriminator_state, indent=2))

print("\n📋 Full Conversation Log:")
for i, entry in enumerate(discriminator_state["context"].get("conversation_log", []), start=1):
    print(f"\n📝 Entry {i}:\n{entry}")


✅ Running Discriminator State
🧠 Discriminator State Result:
{
  "state": "end",
  "file_path": "C:\\Repos\\codecritic\\tests\\notebooks\\system_development\\bad_code.py",
  "context": {
    "file_path": "C:\\Repos\\codecritic\\tests\\notebooks\\system_development\\bad_code.py",
    "source_code": "import sys\nimport os\n\n\ndef add(a: int, b: int) -> int:\n    result = a + b\n    return result\n\n\nclass MyClass:\n    def __init__(self, value: int):\n        self.value = value\n\n    def print_value(self) -> None:\n        print(self.value)\n\n\nadd(1, 2)",
    "score": {
      "name": "linting_score",
      "value": 0.88,
      "components": {
        "ruff": 0.8,
        "mypy": 0.95,
        "black": 1.0,
        "radon": 0.75
      }
    },
    "conversation_log": [
      "- Removed the unused `unused_function` to eliminate dead code and improve clarity.\n- Split the import statement into two separate lines for `sys` and `os` to comply with PEP8 guidelines.\n- Added type hints to t